# Creating Matched Datasets  
I will use T1 structural as an example.   
**train_matched, val_matched and test_matched** are created with T1 surface. I excluded participants with diagnoses of hypertension and artherosclerosis in both positive and negative cases, resulting in smaller datasets.  
train_matched: 138  
val_matched: 40  
test_matched: 20  
**train_matched_excl, val_matched_excl and test_matched_excl** are created with T1 surface, but only excluded negative cases with the diagnoses of the other two diseases. Thus, these datasets are slightly bigger.  
train_matched_excl: 332  
val_matched_excl: 94  
test_matched_excl: 48

In [ ]:
import pandas as pd
import json
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from sklearn.model_selection import train_test_split
import numpy as np
import os
import time
from sklearn import preprocessing
import csv
from matplotlib import cm
from matplotlib.colors import ListedColormap, LinearSegmentedColormap

I converted ethnicities into different categories here for matching

In [ ]:
icd_df = pd.read_csv('{path to the dataset that contains demographic information of patients}') 
icd_df = pd.get_dummies(icd_df, columns=["Ethnic background wide"], prefix="", prefix_sep="")
icd_df.info()

In [ ]:
max_age = icd_df['Age'].max()
min_age = icd_df['Age'].min()
print(max_age)
print(min_age)

In [ ]:
# this step should be before matching (matching only valid patients)
cbv_diagnoses = pd.read_csv('icd_struct.csv') # replace it with your csv 

icd_df.reset_index(inplace=True)

union_eid = pd.Index(cbv_diagnoses['eid']).intersection(icd_df['eid'])

icd_df = icd_df[icd_df['eid'].isin(union_eid)]

len(icd_df)

In [ ]:
variables_for_matching = ['Age', 'Sex_Male', 'Body mass index (BMI)',
                          'Asian or Asian British', 'Black or Black British',
                         'Chinese', 'Mixed', 'Other ethnic group',
                         'White']

Only negative cases with the other two diagnoses are excluded.

In [ ]:
# Filter cases and controls
cases = icd_df[icd_df['I67'] == 1].reset_index(drop=True)
controls = icd_df[(icd_df['I67'] == 0) & (icd_df['I70'] == 0) & (icd_df['I10'] == 0)].reset_index(drop=True)

# Select only the columns for matching but keep 'eid'
cases_for_matching = cases.loc[:, ['eid'] + variables_for_matching]
controls_for_matching = controls.loc[:, ['eid'] + variables_for_matching]

In [ ]:
print(len(cases))

Matching algorithm is defined here.   
Matching is based on "normalized smallest distance".

In [ ]:
# Function to calculate weighted distance
def norm(X_i, X_m, w):
    dx = X_m - X_i
    nan_dims = np.logical_xor(np.isnan(X_m), np.isnan(X_i)).astype(int)
    if w.ndim == 1:
        return np.nansum(dx ** 2 * w, 1) + nan_dims.sum(1)
    else:
        return (dx.dot(w) * dx).sum(1)

# Function to find the smallest m elements
def smallestm(d, m):
    par_idx = np.argpartition(d, m)
    if d[par_idx[:m]].max() < d[par_idx[m]]:  # m < (m+1)th
        return par_idx[:m]
    elif d[par_idx[m]] < d[par_idx[m + 1]].min():  # m+1 < (m+2)th
        return par_idx[:m + 1]
    else:  # mth = (m+1)th = (m+2)th, so increment and recurse
        return smallestm(d, m + 2)

# Function to match a single case to the closest control
def match(X_i, X_m, w):
    d = norm(X_i, X_m, w)
    smallest_idx = smallestm(d, 1)
    smallest_distance = d[smallest_idx]
    return smallest_idx, smallest_distance

In [ ]:
# Initialize an empty DataFrame for matched pairs
matched_pairs = pd.DataFrame(columns=['eid', 'I67'])

# Weights for each variable (can be adjusted)
weights = np.ones(len(variables_for_matching))

# Matching process
for i, case in cases.iterrows():
    case_values = case[variables_for_matching].values.reshape(1, -1).astype(np.float64)  # Exclude 'eid' and ensure float
    control_values = controls[variables_for_matching].values.astype(np.float64)
    idx, dist = match(case_values, control_values, weights)
    
    matched_control = controls_for_matching.iloc[idx[0]]
    matched_pair = pd.DataFrame([
        {'eid': int(case['eid']), 'I67': 1},
        {'eid': int(matched_control['eid']), 'I67': 0}
    ])
    matched_pairs = pd.concat([matched_pairs, matched_pair], ignore_index=True)


print(matched_pairs.head())

In [ ]:
len(matched_pairs)

Get some demorgraphic information about the final cohort.

In [ ]:
merged_df = pd.merge(matched_pairs, icd_df, on='eid')
age_stats = merged_df.groupby('I67_x')['Age'].agg(['mean', 'std']).reset_index()
BMI_stats = merged_df.groupby('I67_x')['Body mass index (BMI)'].agg(['mean', 'std']).reset_index()
sex_stats = merged_df.groupby('I67_x')['Sex_Male'].sum().reset_index()
asian_stats = merged_df.groupby('I67_x')['Asian or Asian British'].sum().reset_index()
black_stats = merged_df.groupby('I67_x')['Black or Black British'].sum().reset_index()
chinese_stats = merged_df.groupby('I67_x')['Chinese'].sum().reset_index()
mixed_stats = merged_df.groupby('I67_x')['Mixed'].sum().reset_index()
other_stats = merged_df.groupby('I67_x')['Other ethnic group'].sum().reset_index()
white_stats = merged_df.groupby('I67_x')['White'].sum().reset_index()
I70 = merged_df.groupby('I67_x')['I70'].sum().reset_index()
I10 = merged_df.groupby('I67_x')['I10'].sum().reset_index()
print('age stats')
print(age_stats)
print('BMI stats')
print(BMI_stats)
print('number of males')
print(sex_stats)
print('asians')
print(asian_stats)
print('blacks')
print(black_stats)
print('chinese')
print(chinese_stats)
print('mixed')
print(mixed_stats)
print('other')
print(other_stats)
print('white')
print(white_stats)
print('I70')
print(I70)
print('I10')
print(I10)

Get the final matched datasets:  
train:val:test = 7:2:1  
train_matched_tem: 324  
val_matched_tem: 92  
test_matched_tem: 46

In [ ]:
# Filter the data into positive and negative patients
positive_cases = matched_pairs[matched_pairs['I67'] == 1]
negative_cases = matched_pairs[matched_pairs['I67'] == 0]

In [ ]:
def stratified_sample(data, train_size, val_size, test_size, random_state=42):
    train_pos_sample = positive_cases.sample(frac=train_size, random_state=random_state)
    train_neg_sample = negative_cases.sample(frac=train_size, random_state=random_state)
    
    remaining_positive_cases = positive_cases.drop(train_pos_sample.index)
    remaining_negative_cases = negative_cases.drop(train_neg_sample.index)
    
    val_pos_sample = remaining_positive_cases.sample(frac=val_size / (val_size + test_size), random_state=random_state)
    val_neg_sample = remaining_negative_cases.sample(frac=val_size / (val_size + test_size), random_state=random_state)
    
    test_pos_sample = remaining_positive_cases.drop(val_pos_sample.index)
    test_neg_sample = remaining_negative_cases.drop(val_neg_sample.index)
    
    train_data = pd.concat([train_pos_sample, train_neg_sample]).sample(frac=1, random_state=random_state).reset_index(drop=True)
    val_data = pd.concat([val_pos_sample, val_neg_sample]).sample(frac=1, random_state=random_state).reset_index(drop=True)
    test_data = pd.concat([test_pos_sample, test_neg_sample]).sample(frac=1, random_state=random_state).reset_index(drop=True)
    
    return train_data, val_data, test_data

# Sample sizes (can be adjusted as needed)
train_size = 0.7
val_size = 0.2
test_size = 0.1

train_data, val_data, test_data = stratified_sample(matched_pairs, train_size, val_size, test_size)

train_data.to_csv('train_matched_tem.csv', index=False)
val_data.to_csv('val_matched_tem.csv', index=False)
test_data.to_csv('test_matched_tem.csv', index=False)